### Load Model

In [ ]:
import subprocess, time, requests
import os
# os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"

process = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "Qwen/Qwen2.5-3B-Instruct",
     "--port", "8000",
     "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.9",
     "--dtype", "half",
    #  "--enforce-eager"],           # disables CUDA graphs + uses eager mode, avoids FA2
    #  "--attention-backend", "xformers"
     ],
    stdout=open("vllm.log", "w"),
    stderr=subprocess.STDOUT
)

for i in range(2):
    try:
        r = requests.get("http://localhost:8000/v1/models")
        if r.status_code == 200:
            print(f"Server ready after {(i+1)*5}s!")
            break
    except:
        pass
    time.sleep(5)
else:
    print("Server didn't start. Checking logs:")


### Test that model works

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")
resp = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct",
    messages=[{"role": "user", "content": "What is 2+2?"}],
    max_tokens=50
)
print(resp.choices[0].message.content)


### Run Program of Thought

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("results", exist_ok=True)

DATASET = 'wikitq'
DATASPLIT = "test"
G_TYPE = "pot"
N_SHOTS = 8
N_SAMPLE = 1
TEMPERATURE = 0.0
ENGINE = 'Qwen/Qwen2.5-3B-Instruct'
API_FILE = "key.txt"
LOG_FILE = f"results/2_{G_TYPE}_{DATASET}_{DATASPLIT}_{N_SHOTS}_{N_SAMPLE}_{ENGINE.replace('/', '_')}_{TEMPERATURE}.log"

os.system(f"""python scripts/execute_pot.py --dataset {DATASET} \
--dataset_split {DATASPLIT} \
--prompt_file templates/prompts/wikitq_pot_ic.txt \
--output_program_file 2_{G_TYPE}_{DATASET}.json \
--n_parallel_prompts 1 \
--n_processes 1 \
--n_shots {N_SHOTS} \
--max_generation_tokens 256 \
--max_api_total_tokens 4096 \
--engine {ENGINE} \
--api_config_file {API_FILE} 2>&1 | tee {LOG_FILE}""")

In [ ]:
os.system(f"""python scripts/execute_pot_finqa.py \
    --engine Qwen/Qwen2.5-3B-Instruct \
    --dataset finqa \
    --dataset_split test \
    --max_generation_tokens 256 \
    --max_api_total_tokens 4001 \
    --n_processes 1 \
    --output_program_file pot_finqa_test_qwen3b.json""")

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.makedirs("results", exist_ok=True)

os.system(f"""python scripts/execute_pot_tatqa.py \
    --engine Qwen/Qwen2.5-3B-Instruct \
    --dataset tatqa \
    --dataset_split validation \
    --max_generation_tokens 256 \
    --max_api_total_tokens 4096 \
    --n_processes 1 \
    --output_program_file pot_tatqa_validation_qwen3b.json 2>&1 | tee results/pot_tatqa_validation_qwen3b.log""")

### Run Binder

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.makedirs("results", exist_ok=True)

DATASET = "wikitq"
DATASPLIT = "test"
G_TYPE = "nsql"
N_SHOTS = 8
N_SAMPLE = 1
TEMPERATURE = 0.0
ENGINE = "Qwen/Qwen2.5-3B-Instruct"
API_FILE = "key.txt"

PROG_FILE = f"{G_TYPE}_{DATASET}_{DATASPLIT}_{N_SHOTS}_{N_SAMPLE}_{ENGINE.replace('/', '_')}_{TEMPERATURE}.json"
EXEC_FILE = f"{G_TYPE}_{DATASET}_{DATASPLIT}_{N_SHOTS}_{N_SAMPLE}_{ENGINE.replace('/', '_')}_{TEMPERATURE}_exec.json"
LOG_FILE = f"results/{G_TYPE}_{DATASET}_{DATASPLIT}_{ENGINE.replace('/', '_')}.log"

# Step 1: Generate binder programs
os.system(f"""python scripts/annotate_binder_program.py --dataset {DATASET} \
--dataset_split {DATASPLIT} \
--prompt_file templates/prompts/wikitq_nsqlcot_ic.txt \
--system_prompt_file templates/prompts/wikitq_nsqlcot_system.txt \
--output_program_file {PROG_FILE} \
--n_parallel_prompts 1 \
--n_processes 1 \
--n_shots {N_SHOTS} \
--max_generation_tokens 256 \
--max_api_total_tokens 4096 \
--temperature {TEMPERATURE} \
--sampling_n {N_SAMPLE} \
--engine {ENGINE} \
--api_config_file {API_FILE} 2>&1 | tee {LOG_FILE}""")

# Step 2: Execute binder programs
os.system(f"""python scripts/execute_binder_program.py --dataset {DATASET} \
--dataset_split {DATASPLIT} \
--qa_retrieve_pool_file templates/qa_retrieve_pool/qa_retrieve_pool.json \
--input_program_file {PROG_FILE} \
--output_program_execution_file {EXEC_FILE} \
--vote_method count \
--engine {ENGINE} \
--n_processes 1 \
--api_config_file {API_FILE} \
--use_cot 2>&1 | tee -a {LOG_FILE}""")
